# 07 — Ablation and qualitative evaluation

Brings the four runs together:
- Trainable parameter counts per regime.
- In-domain (synthetic CFRP) best val Dice.
- Cross-domain evaluation on the real Chalmers scan — purely qualitative since it has no GT in the Zenodo record, plus the SiC-SiC held-out volume (if you've ingested the ACDC data).
- Fibre continuity metric comparison.
- Slice triptychs for each regime.

In [ ]:
%load_ext autoreload
%autoreload 2
import warnings; warnings.filterwarnings('ignore')

import json
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from skimage.transform import resize as sk_resize

from cfrp_medsam2.data import NpzVolume, mask_to_bbox
from cfrp_medsam2.model import SegModel, ModelConfig
from cfrp_medsam2.lora import LoRAConfig, inject_lora
from cfrp_medsam2.eval import summarize
from cfrp_medsam2.viz import slice_triptych, overlay_slice

REPO = Path('..').resolve()
device = 'cuda' if torch.cuda.is_available() else 'cpu'

## 1. Ablation table from saved logs

In [ ]:
rows = []
for regime in ['zero_shot', 'lora', 'conv_lora', 'full_ft']:
    row = {'regime': regime}
    if regime == 'zero_shot':
        j = REPO / 'logs' / 'zero_shot_metrics.json'
        if j.exists():
            m = json.load(open(j))
            row.update({'best_val_dice': m.get('dice_3d'), 'dice_slice_mean': m.get('dice_slice_mean'), 'fibre_continuity': m.get('fibre_continuity')})
    else:
        csv = REPO / 'logs' / f'{regime}.csv'
        if csv.exists():
            df = pd.read_csv(csv)
            if len(df):
                row.update({'best_val_dice': df['val_dice'].max(), 'last_epoch': int(df['epoch'].max())})
    rows.append(row)
tbl = pd.DataFrame(rows)
tbl

## 2. Trainable parameters per regime (as fractions of the backbone)

In [ ]:
from cfrp_medsam2.lora import trainable_param_summary

ckpt = REPO / 'checkpoints' / 'sam2.1_hiera_tiny.pt'
pct = {}
for regime in ['lora', 'conv_lora', 'full_ft']:
    m = SegModel(ModelConfig(backend='medsam2', checkpoint=str(ckpt), device=device))
    if regime == 'lora':
        inject_lora(m, LoRAConfig(rank=8, target_substrings=('qkv', 'q_proj', 'k_proj', 'v_proj', 'out_proj', 'proj'), exclude_substrings=('mlp', 'obj_ptr', 'mask_decoder.iou_prediction_head')))
    elif regime == 'conv_lora':
        inject_lora(m, LoRAConfig(rank=8, use_conv=True, target_substrings=('qkv', 'q_proj', 'k_proj', 'v_proj', 'out_proj', 'proj'), exclude_substrings=('mlp', 'obj_ptr', 'mask_decoder.iou_prediction_head')))
    else:
        for p in m.parameters():
            p.requires_grad = True
    s = trainable_param_summary(m)
    pct[regime] = {'trainable': s['trainable'], 'total': s['total'], 'pct': 100 * s['trainable'] / s['total']}
    del m; torch.cuda.empty_cache()
pd.DataFrame(pct).T

## 3. Qualitative: zero-shot vs LoRA vs Conv-LoRA on the middle slice

In [ ]:
val_npz = REPO / 'data' / 'processed' / 'toy' / 'val_00.npz'
vol = NpzVolume.load(val_npz)
S = 512
imgs_s = np.stack([sk_resize(s, (S, S), order=1, preserve_range=True, anti_aliasing=True).astype(np.uint8) for s in vol.imgs])
gts_s = np.stack([sk_resize(s, (S, S), order=0, preserve_range=True, anti_aliasing=False).astype(np.uint8) for s in vol.gts])
z = len(imgs_s) // 2
print('middle slice z =', z)

def predict_slice(checkpoint_path, lora_cfg=None):
    m = SegModel(ModelConfig(backend='medsam2', checkpoint=str(ckpt), device=device))
    if lora_cfg is not None:
        inject_lora(m, lora_cfg)
    if checkpoint_path and Path(checkpoint_path).exists():
        state = torch.load(checkpoint_path, map_location=device, weights_only=False)['state_dict']
        m.load_state_dict(state, strict=False)
    m.eval()
    rgb = np.repeat(imgs_s[z][None], 3, axis=0).astype(np.float32) / 255.0
    img = torch.from_numpy(rgb).unsqueeze(0).to(device)
    bb = mask_to_bbox(gts_s[z])
    box = torch.tensor(bb, dtype=torch.float32, device=device)
    with torch.no_grad():
        logits = m.forward_slice(img, boxes=[box])
    pred = (torch.sigmoid(logits[0]).cpu().numpy() > 0.5).astype(np.uint8)
    del m; torch.cuda.empty_cache()
    return pred

preds = {}
preds['zero_shot'] = predict_slice(None, None)
preds['lora'] = predict_slice(REPO / 'checkpoints' / 'lora_best.pt', LoRAConfig(rank=8, target_substrings=('qkv', 'q_proj', 'k_proj', 'v_proj', 'out_proj', 'proj'), exclude_substrings=('mlp', 'obj_ptr', 'mask_decoder.iou_prediction_head')))
preds['conv_lora'] = predict_slice(REPO / 'checkpoints' / 'conv_lora_best.pt', LoRAConfig(rank=8, use_conv=True, target_substrings=('qkv', 'q_proj', 'k_proj', 'v_proj', 'out_proj', 'proj'), exclude_substrings=('mlp', 'obj_ptr', 'mask_decoder.iou_prediction_head')))

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
axes[0].imshow(imgs_s[z], cmap='gray'); axes[0].set_title('image'); axes[0].axis('off')
axes[1].imshow(overlay_slice(imgs_s[z], gts_s[z])); axes[1].set_title('GT'); axes[1].axis('off')
for ax, (k, p) in zip(axes[2:], list(preds.items())[:2]):
    ax.imshow(overlay_slice(imgs_s[z], p)); ax.set_title(k); ax.axis('off')
plt.tight_layout(); plt.show()

## 4. Cross-domain: real CFRP scan (no GT — qualitative only)

In [ ]:
real_npz = REPO / 'data' / 'processed' / 'cfrp' / 'real_chunk.npz'
if real_npz.exists():
    vr = NpzVolume.load(real_npz)
    zs = len(vr.imgs) // 2
    imgr = sk_resize(vr.imgs[zs], (512, 512), order=1, preserve_range=True, anti_aliasing=True).astype(np.uint8)
    rgbr = np.repeat(imgr[None], 3, axis=0).astype(np.float32) / 255.0
    m = SegModel(ModelConfig(backend='medsam2', checkpoint=str(ckpt), device=device))
    inject_lora(m, LoRAConfig(rank=8, target_substrings=('qkv', 'q_proj', 'k_proj', 'v_proj', 'out_proj', 'proj'), exclude_substrings=('mlp', 'obj_ptr', 'mask_decoder.iou_prediction_head')))
    lora_ckpt = REPO / 'checkpoints' / 'lora_best.pt'
    if lora_ckpt.exists():
        state = torch.load(lora_ckpt, map_location=device, weights_only=False)['state_dict']
        m.load_state_dict(state, strict=False)
    m.eval()
    # Heuristic: prompt box covers the whole interior ROI of the scan.
    with torch.no_grad():
        img = torch.from_numpy(rgbr).unsqueeze(0).to(device)
        box = torch.tensor([50.0, 50.0, 462.0, 462.0], device=device)
        logits = m.forward_slice(img, boxes=[box])
        predr = (torch.sigmoid(logits[0]).cpu().numpy() > 0.5).astype(np.uint8)
    fig, axes = plt.subplots(1, 2, figsize=(8, 4))
    axes[0].imshow(imgr, cmap='gray'); axes[0].set_title('real CFRP slice'); axes[0].axis('off')
    axes[1].imshow(overlay_slice(imgr, predr)); axes[1].set_title('LoRA prediction'); axes[1].axis('off')
    plt.tight_layout(); plt.show()
else:
    print('no real_chunk.npz — skip')

## 5. Fibre continuity comparison (3D Dice + continuity ratio per regime)

In [ ]:
def full_volume_eval(regime, lora_cfg=None, ckpt_name=None):
    m = SegModel(ModelConfig(backend='medsam2', checkpoint=str(ckpt), device=device))
    if lora_cfg is not None:
        inject_lora(m, lora_cfg)
    if ckpt_name:
        p = REPO / 'checkpoints' / ckpt_name
        if p.exists():
            state = torch.load(p, map_location=device, weights_only=False)['state_dict']
            m.load_state_dict(state, strict=False)
    m.eval()
    preds = np.zeros_like(gts_s)
    with torch.no_grad():
        for zz in range(len(imgs_s)):
            rgb = np.repeat(imgs_s[zz][None], 3, axis=0).astype(np.float32) / 255.0
            img = torch.from_numpy(rgb).unsqueeze(0).to(device)
            bb = mask_to_bbox(gts_s[zz])
            if bb is None:
                continue
            box = torch.tensor(bb, dtype=torch.float32, device=device)
            logits = m.forward_slice(img, boxes=[box])
            preds[zz] = (torch.sigmoid(logits[0]).cpu().numpy() > 0.5).astype(np.uint8)
    out = summarize(preds, gts_s)
    del m; torch.cuda.empty_cache()
    return out

base = ('qkv', 'q_proj', 'k_proj', 'v_proj', 'out_proj', 'proj')
excl = ('mlp', 'obj_ptr', 'mask_decoder.iou_prediction_head')
rows = {
    'zero_shot': full_volume_eval('zero_shot'),
    'lora':       full_volume_eval('lora',       LoRAConfig(rank=8, target_substrings=base, exclude_substrings=excl),                  'lora_best.pt'),
    'conv_lora':  full_volume_eval('conv_lora',  LoRAConfig(rank=8, use_conv=True, target_substrings=base, exclude_substrings=excl),  'conv_lora_best.pt'),
}
pd.DataFrame(rows).T

## Takeaways
- Both LoRA variants should clear zero-shot at <1% parameter cost.
- Conv-LoRA's local prior is most visible on `fibre_continuity` — the metric that captures the slice-wise-breakage failure mode.
- Full fine-tune is the upper bound but demands much more compute; on a single 24 GB GPU the LoRA variants are the practical default.